In [1]:
!pip install numpy scipy casadi pyserial matplotlib pyserial
print('OK')

OK



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\PC\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
"""
Koopman Closed-Loop v4b-fix — IMPLEMENTACIÓN PLANTA FÍSICA REAL (100% Koopman)
================================================================================
VERSIÓN v4b-fix: corrige el sobrepico de v4b original.

Cambios respecto a v4b original
--------------------------------
1. Anti-windup mejorado (freeze durante transitorio):
   - El integrador se CONGELA mientras |error_h2| > DEADBAND_H2.
   - Antes solo se congelaba cuando |error| < DEADBAND_H2, lo que permitía
     que z_int se cargara libremente en los pasos grandes (step 10→20 cm).

2. Límite de integrador más ajustado:
   - Z_INT_LIMIT: 0.03 m·s → 0.015 m·s (evita windup severo en step grande)

3. Peso integrador aumentado en el MPC:
   - KI_WEIGHT: 50 → 80 (penaliza más el estado acumulado en el horizonte)

4. Peso terminal distribuido:
   - Antes: solo el último paso tenía wk=10.
   - Ahora: los últimos 3 pasos tienen wk=5, el resto wk=1.
   - Evita que el MPC ignore el costo de z_int en el horizonte medio.

5. Reset de warm-start MPC consistente:
   - Al cambiar referencia, el warm-start se construye con z_int=0 explícito,
     no con el z_int residual del paso anterior.

Todo lo demás es idéntico a v4b.
"""

import numpy as np
from scipy.optimize import least_squares
import scipy.linalg as la
import casadi as ca
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import time
import serial
import serial.tools.list_ports
import threading
import queue
import sys
import csv
from datetime import datetime

warnings.filterwarnings('ignore')

output_dir = Path('./outputs_real')
output_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# PARÁMETROS FÍSICOS — CALIBRADOS DESDE MEDICIÓN REAL
# ============================================================================
P_NOM = {
    'Atank'   : 3.02e-3,
    'rho'     : 997.0,
    'eta'     : 8.9e-4,
    'g'       : 9.81,
    'alphao1' : 0.0271, 'Do1': 10e-3,
    'alphao2' : 0.0271, 'A2' : 7.85e-5,
    'alphao3' : 0.0271, 'Do3': 10e-3,
    'alpha120': 0.3038, 'D12': 10e-3, 'A12': 7.85e-5, 'lambdac12': 24000,
    'alpha230': 0.1344, 'D23': 10e-3, 'A23': 7.85e-5, 'lambdac23': 29600,
    'qi1max'  : 2.69e-5,
    'qi3max'  : 2.41e-5,
}

# ============================================================================
# PARÁMETROS DE SIMULACIÓN / CONTROL
# ============================================================================
Ts         = 2.0
T_sim      = 800
Nsim       = int(T_sim / Ts)
EPS        = 1e-10
EPS_PSI    = 1e-8
SETTLING_S = 60

SERIAL_PORT    = 'COM7'
BAUD_RATE      = 115200
SERIAL_TIMEOUT = 3.0

MAX_DELTA_H = 0.05   # m — filtro anti-outlier ultrasónico

REF_STEPS = [
    (0,              0.10),
    (int(Nsim*0.33), 0.20),
    (int(Nsim*0.66), 0.15),
]

# ============================================================================
# PARÁMETROS DEL OBSERVADOR E INTEGRADOR
# ============================================================================
Q_OBSERVER  = 0.1

# ── CORRECCIONES anti-sobrepico ──────────────────────────────────────────────
KI_WEIGHT   = 80.0          # subido de 50 → 80 (penaliza más z_int en MPC)
DEADBAND_H2 = 0.03          # m — sin cambio
Z_INT_LIMIT = 0.015         # m·s — reducido de 0.03 → 0.015

# Rampa de referencia: suaviza steps grandes para evitar saturación de u1
# El MPC ve la referencia "rampeada" durante el transitorio, no el escalón.
# Tiempo de rampa en segundos para un step de 10 cm (proporcional al step).
T_RAMP_PER_10CM = 40.0      # s — rampa de 40 s para un step de 10 cm
# ─────────────────────────────────────────────────────────────────────────────

# ============================================================================
# MODELO INTERNO
# ============================================================================
def nl_ode(x, u, p=P_NOM):
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = p['qi1max'] * np.clip(u[0], 0, 1)
    qi3 = p['qi3max'] * np.clip(u[1], 0, 1)
    qo1 = p['alphao1'] * (p['Do1']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h1)
    qo2 = p['alphao2'] * p['A2']                    * np.sqrt(2 * p['g'] * h2)
    qo3 = p['alphao3'] * (p['Do3']**2 * np.pi / 4) * np.sqrt(2 * p['g'] * h3)
    dh12 = h1 - h2
    l12  = p['D12'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS)
    a12  = p['alpha120'] * np.tanh(2 * l12 / p['lambdac12'])
    q12  = a12 * p['A12'] * np.sqrt(2 * p['g'] * abs(dh12) + EPS) * np.sign(dh12 + 1e-15)
    dh23 = h2 - h3
    l23  = p['D23'] * p['rho'] / p['eta'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS)
    a23  = p['alpha230'] * np.tanh(2 * l23 / p['lambdac23'])
    q23  = a23 * p['A23'] * np.sqrt(2 * p['g'] * abs(dh23) + EPS) * np.sign(dh23 + 1e-15)
    return np.array([
        (qi1 - q12 - qo1) / p['Atank'],
        (q12 - q23 - qo2) / p['Atank'],
        (qi3 + q23 - qo3) / p['Atank'],
    ])

def rk4(x, u, p=P_NOM, dt=Ts):
    k1 = nl_ode(x, u, p); k2 = nl_ode(x + dt/2*k1, u, p)
    k3 = nl_ode(x + dt/2*k2, u, p); k4 = nl_ode(x + dt*k3, u, p)
    return np.clip(x + dt/6*(k1 + 2*k2 + 2*k3 + k4), 0, 0.27)

def psi(x):
    h1, h2, h3 = x[0], x[1], x[2]
    return np.array([
        np.sqrt(max(h1, EPS_PSI)),
        np.sqrt(max(h2, EPS_PSI)),
        np.sqrt(max(h3, EPS_PSI)),
        h1 * h2, h2 * h3, h1 * h3,
        h1, h3,
    ], dtype=float)

def compute_ss(h2_ref, p=P_NOM):
    w_reg = 0.05
    def eqs(v):
        h1, h3, u1, u3 = v
        f = nl_ode([max(h1, 1e-4), h2_ref, max(h3, 1e-4)],
                   [np.clip(u1, 0, 1), np.clip(u3, 0, 1)], p)
        return np.concatenate([f, [w_reg * u3]])
    guesses = [
        (h2_ref*1.4, h2_ref*0.8, 0.5, 0.0), (h2_ref*1.6, h2_ref*0.7, 0.7, 0.0),
        (h2_ref*2.0, h2_ref*0.8, 0.9, 0.0), (h2_ref*2.5, h2_ref*0.8, 0.7, 0.0),
        (h2_ref*2.5, h2_ref*0.8, 0.75, 0.0),(h2_ref*1.8, h2_ref*0.85, 0.85, 0.0),
        (h2_ref*1.3, h2_ref*0.9, 1.0, 0.1), (h2_ref*1.2, h2_ref*1.0, 1.0, 0.3),
        (h2_ref*1.1, h2_ref*1.1, 1.0, 0.5), (h2_ref*1.0, h2_ref*1.2, 1.0, 0.8),
    ]
    best, best_res = None, 1e10
    for g in guesses:
        try:
            s = least_squares(eqs, list(g),
                              bounds=([0.01, 0.01, 0.0, 0.0], [0.27, 0.27, 1.0, 1.0]),
                              max_nfev=5000)
            phys_res = np.linalg.norm(s.fun[:3])
            if s.success and phys_res < best_res:
                best_res = phys_res; best = s.x
        except: pass
    if best is not None and best_res < 1e-4:
        return (np.array([best[0], h2_ref, best[1]]),
                np.array([np.clip(best[2], 0, 1), np.clip(best[3], 0, 1)]))
    return None, None

# ============================================================================
# COMUNICACIÓN SERIAL CON ESP32
# ============================================================================
class ESP32Interface:
    def __init__(self, port=SERIAL_PORT, baud=BAUD_RATE):
        self.port = port; self.baud = baud; self.ser = None
        self.rx_queue    = queue.Queue(maxsize=20)
        self._stop_event = threading.Event()
        self._last_meas  = None
        self._lock       = threading.Lock()
        self.outlier_count = 0

    def connect(self):
        print(f"\nConectando a {self.port} @ {self.baud} baud...")
        try:
            self.ser = serial.Serial(self.port, self.baud,
                                     timeout=SERIAL_TIMEOUT, write_timeout=1.0)
            time.sleep(2.0)
            self.ser.reset_input_buffer()
            print(f"  ✓ Conectado a {self.port}")
        except serial.SerialException as e:
            print(f"  ✗ Error: {e}")
            for p in serial.tools.list_ports.comports():
                print(f"    {p.device} — {p.description}")
            raise
        self._reader_thread = threading.Thread(target=self._reader_loop, daemon=True)
        self._reader_thread.start()

    def _reader_loop(self):
        while not self._stop_event.is_set():
            try:
                if self.ser.in_waiting:
                    line = self.ser.readline().decode('utf-8', errors='ignore').strip()
                    if line:
                        self.rx_queue.put_nowait(line)
            except Exception: pass
            time.sleep(0.005)

    def get_measurement(self, timeout=3.0):
        """Devuelve (h1, h2_real, h3) en metros."""
        deadline = time.time() + timeout
        while time.time() < deadline:
            try:
                line = self.rx_queue.get(timeout=0.1)
                line = line.replace('\x00', '').replace('\0', '').strip()
                line = ''.join(c for c in line if c.isprintable())
                if not line: continue
                parts = line.split(',')
                if len(parts) != 3: continue
                try:
                    h1      = max(float(parts[0].strip()), 0.0)
                    h2_real = max(float(parts[1].strip()), 0.0)
                    h3      = max(float(parts[2].strip()), 0.0)
                except (ValueError, OverflowError): continue

                with self._lock:
                    if self._last_meas is not None:
                        h1p, h2p, h3p = self._last_meas
                        if abs(h1 - h1p) > MAX_DELTA_H:
                            h1 = h1p; self.outlier_count += 1
                        if abs(h2_real - h2p) > MAX_DELTA_H:
                            h2_real = h2p; self.outlier_count += 1
                        if abs(h3 - h3p) > MAX_DELTA_H:
                            h3 = h3p; self.outlier_count += 1
                    self._last_meas = (h1, h2_real, h3)

                return h1, h2_real, h3

            except queue.Empty: pass

        with self._lock:
            if self._last_meas is not None:
                print("  ⚠ Timeout serial — usando última medición")
                return self._last_meas
        raise TimeoutError("No se recibieron datos del ESP32")

    def send_control(self, u1: float, u2: float):
        u1 = float(np.clip(u1, 0.0, 1.0)); u2 = float(np.clip(u2, 0.0, 1.0))
        try:
            self.ser.write(f"{u1:.4f},{u2:.4f}\n".encode('utf-8'))
        except Exception as e:
            print(f"  ✗ Error al enviar: {e}")

    def stop_pumps(self):
        self.send_control(0.0, 0.0)
        print("  Bombas apagadas.")

    def close(self):
        self._stop_event.set()
        self.stop_pumps()
        time.sleep(0.5)
        if self.ser and self.ser.is_open:
            self.ser.close()
        print("  Puerto serial cerrado.")

# ============================================================================
# ENTRENAMIENTO EDMD
# ============================================================================
def train_edmd(verbose=True):
    if verbose:
        print("\nEntrenando EDMD (parámetros nominales)...")
    rng = np.random.default_rng(42)
    z_k, u_k, z_k1 = [], [], []

    for h2r in [0.10, 0.12, 0.15, 0.18, 0.20, 0.22]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None: continue
        for pert in [0.95, 0.90, 0.98, 0.85, 0.92, 0.88]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.05, 0.70)
            for _ in range(170):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.35*err, -0.35, 0.35) + rng.uniform(-0.15, 0.15)
                u[1] += rng.uniform(-0.05, 0.05); u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4(x, u); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is None: continue
        for pert in [0.60, 0.70, 0.75, 1.25, 1.35]:
            x = x_ss.copy(); x[1] = np.clip(h2r * pert, 0.05, 0.70)
            for _ in range(100):
                err = x[1] - h2r; u = u_ss.copy()
                u[0] += np.clip(-0.6*err, -0.6, 0.6) + rng.uniform(-0.1, 0.1)
                u = np.clip(u, 0, 1)
                z = psi(x); xn = rk4(x, u); zn = psi(xn)
                z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    for (h2f, h2t) in [(0.10, 0.20), (0.20, 0.15), (0.15, 0.10)]:
        xf, uf = compute_ss(h2f); xt, _ = compute_ss(h2t)
        if xf is None or xt is None: continue
        x = xf.copy()
        for _ in range(200):
            err = x[1] - h2t; u = uf.copy()
            u[0] += np.clip(-0.5*err, -0.5, 0.5) + rng.uniform(-0.08, 0.08)
            u = np.clip(u, 0, 1)
            z = psi(x); xn = rk4(x, u); zn = psi(xn)
            z_k.append(z); u_k.append(u); z_k1.append(zn); x = xn

    Z = np.array(z_k); U = np.array(u_k); Zn = np.array(z_k1)
    nz, nu = Z.shape[1], U.shape[1]
    Phi = np.hstack([Z, U]); lam = 1e-4
    Theta = np.linalg.solve(Phi.T @ Phi + lam * np.eye(nz + nu), Phi.T @ Zn)
    A_raw = Theta[:nz, :].T; B_d = Theta[nz:, :].T
    vals, vecs = np.linalg.eig(A_raw); delta = 1e-4
    vals_s = np.where(np.abs(vals) >= 1.0, vals / np.abs(vals) * (1.0 - delta), vals)
    A_d = np.real(vecs @ np.diag(vals_s) @ np.linalg.inv(vecs))
    rmse = np.sqrt(np.mean((Z @ A_d.T + U @ B_d.T - Zn)**2))
    eigs = np.abs(np.linalg.eigvals(A_d))
    if verbose:
        print(f"  {len(Z)} muestras de entrenamiento")
        print(f"  RMSE one-step: {rmse:.2e}")
        print(f"  |λ| ∈ [{eigs.min():.5f}, {eigs.max():.5f}] ✓")
    return A_d, B_d, nz, nu

# ============================================================================
# DISEÑO DEL OBSERVADOR
# ============================================================================
def design_observer(A_d, nz):
    C_obs = np.zeros((4, nz))
    C_obs[0, 0] = 1.0   # √h1
    C_obs[1, 2] = 1.0   # √h3
    C_obs[2, 6] = 1.0   # h1
    C_obs[3, 7] = 1.0   # h3
    m = C_obs.shape[0]
    print(f"\nDiseñando ganancia L (DARE con Q={Q_OBSERVER}*I)...")
    for q_try in [Q_OBSERVER, 0.05, 0.5, 1.0, 5.0, 10.0]:
        try:
            Q_obs = q_try * np.eye(nz)
            X = la.solve_discrete_are(A_d.T, C_obs.T, Q_obs, np.eye(m))
            L_d = np.linalg.solve(C_obs @ X @ C_obs.T + np.eye(m), C_obs @ X @ A_d.T).T
            max_eig = np.abs(np.linalg.eigvals(A_d - L_d @ C_obs)).max()
            if max_eig < 1.0 and np.abs(L_d).max() > 0.01:
                print(f"  q={q_try}: max|λ(A-LC)|={max_eig:.5f}, max|L|={np.abs(L_d).max():.4f} ✓")
                print(f"  τ_decay = {-Ts/np.log(max_eig):.1f} s")
                return L_d, C_obs
        except: pass
    raise RuntimeError("No se pudo diseñar L.")

# ============================================================================
# CONSTRUCCIÓN DEL MPC CON INTEGRADOR (v4b-fix)
# ============================================================================
def build_mpc_with_integrator(A_d, B_d, nz, nu):
    print(f"\nConstruyendo KoopmanMPC v4b-fix con integrador (ki={KI_WEIGHT})...")
    N_mpc = 20; q_mpc = 500
    R1 = np.diag([0.1, 1.0]); R2 = np.diag([0.1, 0.1])
    umin = np.array([0.0, 0.0]); umax = np.array([1.0, 1.0])
    nz_aug = nz + 1

    # ── CORRECCIÓN: índices de los últimos pasos con peso elevado ───────────
    # Distribuimos el énfasis terminal en los 3 últimos pasos (wk=5)
    # en lugar de solo el último (wk=10). Reduce el pico de control.
    N_TERMINAL = 3
    W_TERMINAL = 5
    # ────────────────────────────────────────────────────────────────────────

    Z_sym  = ca.SX.sym('Z',  nz_aug, N_mpc+1)
    U_sym  = ca.SX.sym('U',  nu,     N_mpc)
    z0_sym = ca.SX.sym('z0', nz_aug)
    yr_sym = ca.SX.sym('yr')
    up_sym = ca.SX.sym('up', nu)
    A_cas  = ca.DM(A_d); B_cas = ca.DM(B_d)

    J = 0; g = []; lbg = []; ubg = []
    g += [Z_sym[:, 0] - z0_sym]; lbg += [0.0]*nz_aug; ubg += [0.0]*nz_aug

    for k in range(N_mpc):
        zk    = Z_sym[:nz, k];   zk1   = Z_sym[:nz, k+1]
        zint  = Z_sym[nz, k];    zint1 = Z_sym[nz, k+1]
        g    += [zk1 - (A_cas @ zk + B_cas @ U_sym[:, k])]
        lbg  += [0.0]*nz; ubg += [0.0]*nz
        h2_est_k = zk[1]**2
        g    += [zint1 - (zint + (yr_sym - h2_est_k) * Ts)]
        lbg  += [0.0]; ubg += [0.0]
        h2_k  = zk1[1]**2
        ey    = h2_k - yr_sym
        du    = U_sym[:, 0] - up_sym if k == 0 else U_sym[:, k] - U_sym[:, k-1]

        # Peso terminal distribuido en los últimos N_TERMINAL pasos
        wk = W_TERMINAL if k >= (N_mpc - N_TERMINAL) else 1

        J    += (wk * q_mpc * ey**2
                 + KI_WEIGHT * zint**2
                 + ca.mtimes(U_sym[:, k].T, ca.DM(R1) @ U_sym[:, k])
                 + ca.mtimes(du.T,          ca.DM(R2) @ du))

    w_sym   = ca.vertcat(ca.reshape(Z_sym, nz_aug*(N_mpc+1), 1),
                         ca.reshape(U_sym, nu*N_mpc, 1))
    par_sym = ca.vertcat(z0_sym, yr_sym, up_sym)
    lbw = [-ca.inf]*(nz_aug*(N_mpc+1)) + list(np.tile(umin, N_mpc))
    ubw = [ ca.inf]*(nz_aug*(N_mpc+1)) + list(np.tile(umax, N_mpc))
    nlp  = {'x': w_sym, 'f': J, 'g': ca.vertcat(*g), 'p': par_sym}
    opts = {'ipopt.print_level': 0, 'ipopt.max_iter': 500,
            'ipopt.tol': 1e-5, 'ipopt.acceptable_tol': 1e-4, 'print_time': 0}
    ki_str = str(KI_WEIGHT).replace('.', 'p')
    solver = ca.nlpsol(f'kmpc_v4b_fix_{ki_str}', 'ipopt', nlp, opts)
    print(f"  KoopmanMPC v4b-fix ✓  (nz_aug={nz_aug})")
    print(f"  DEADBAND_H2={DEADBAND_H2*100:.1f}cm | Z_INT_LIMIT=±{Z_INT_LIMIT*100:.1f}cm·s")
    print(f"  Peso terminal: wk={W_TERMINAL} en últimos {N_TERMINAL} pasos")
    return solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz_aug, nu

# ============================================================================
# LOOP DE CONTROL EN TIEMPO REAL
# ============================================================================
def run_real_experiment(esp, A_d, B_d, L_d, C_obs,
                        solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg,
                        nz_aug, nu, nz, SS_NOM):

    yref_v = np.zeros(Nsim)
    for (k_start, ref_val) in REF_STEPS:
        yref_v[k_start:] = ref_val
    ref_change_times_s = [k * Ts for k, _ in REF_STEPS[1:]]

    print("\nEsperando primera medición del ESP32...")
    h1_meas, h2_real, h3_meas = esp.get_measurement()
    print(f"  Primera medición: h1={h1_meas*100:.2f}cm | "
          f"h2_real={h2_real*100:.2f}cm | h3={h3_meas*100:.2f}cm")

    x_init    = np.array([h1_meas, h2_real, h3_meas])
    z_koopman = psi(x_init)
    z_int     = 0.0
    z_aug     = np.append(z_koopman, z_int)
    uprev     = np.array([0.0, 0.0])
    w0_mpc    = None
    yr_prev   = yref_v[0]

    # ── flag de transitorio ─────────────────────────────────────────────────
    in_transient = True          # arrancamos en transitorio
    transient_counter = 0        # pasos desde el último cambio de referencia
    TRANSIENT_STEPS   = 30       # ~60 s para declarar steady-state

    # ── Rampa de referencia suavizada ──────────────────────────────────────
    # yr_ramp: referencia que ve el MPC — sube gradualmente desde yr_ramp_start
    # hasta yr_target. Evita saturar u1 en steps grandes (0→10, 10→20 cm).
    yr_ramp_start  = h2_real           # nivel real medido al arrancar
    yr_target      = yref_v[0]         # referencia final del primer tramo
    yr_ramp        = yr_ramp_start     # valor actual de la rampa
    ramp_active    = True              # arrancamos siempre con rampa
    # ────────────────────────────────────────────────────────────────────────

    T_hist  = np.arange(Nsim + 1) * Ts
    H_meas  = np.zeros((2, Nsim + 1))
    H2_REAL = np.zeros(Nsim + 1)
    X_hat   = np.zeros((3, Nsim + 1))
    Z_INT   = np.zeros(Nsim + 1)
    U_hist  = np.zeros((2, Nsim))
    step_ms = []

    H_meas[0, 0]  = h1_meas
    H_meas[1, 0]  = h3_meas
    H2_REAL[0]    = h2_real
    X_hat[:, 0]   = x_init
    Z_INT[0]      = z_int

    print(f"\n{'='*72}")
    print(f"  EXPERIMENTO v4b-fix INICIADO — {datetime.now().strftime('%H:%M:%S')}")
    print(f"  KI_WEIGHT={KI_WEIGHT} | Z_INT_LIMIT=±{Z_INT_LIMIT*100:.1f}cm·s | "
          f"DEADBAND={DEADBAND_H2*100:.1f}cm")
    print(f"  Anti-windup: integrador CONGELADO durante transitorio")
    print(f"{'='*72}")
    print(f"  {'t[s]':>6} | {'h1[cm]':>7} | {'ĥ2[cm]':>7} | {'h2r[cm]':>7} | "
          f"{'eobs[cm]':>8} | {'ref[cm]':>7} | {'u1':>5} | {'ms':>5} | {'trns':>5}")
    print(f"  {'-'*82}")

    t_experiment_start = time.time()

    for k in range(Nsim):
        t_step_start = time.time()
        yr = yref_v[k]

        # ── Detectar cambio de referencia ───────────────────────────────────
        if yr != yr_prev:
            # Reset integrador y marcar inicio de transitorio
            z_int = 0.0; Z_INT[k] = 0.0
            in_transient = True
            transient_counter = 0

            # Iniciar rampa desde el nivel estimado actual hasta la nueva ref
            yr_ramp_start = x_hat[1] if k > 0 else h2_real
            yr_target     = yr
            yr_ramp       = yr_ramp_start
            ramp_active   = True

            # Warm-start MPC con z_int=0 explícito
            if yr in SS_NOM:
                z_ss_new = psi(SS_NOM[yr][0]); u_ss_new = SS_NOM[yr][1]
                z_aug_ss = np.append(z_ss_new, 0.0)   # z_int=0 explícito
                Z0 = np.tile(z_aug_ss, (N_mpc+1, 1)); U0 = np.tile(u_ss_new, (N_mpc, 1))
                w0_mpc = np.concatenate([Z0.flatten(), U0.flatten()])
            else:
                w0_mpc = None
            print(f"\n  ▶ Cambio referencia: {yr_prev*100:.0f}→{yr*100:.0f} cm "
                  f"| Reset z_int=0 | rampa {yr_ramp_start*100:.1f}→{yr*100:.1f} cm")
        yr_prev = yr

        # ── Avanzar rampa de referencia ─────────────────────────────────────
        # La rampa sube/baja a una tasa proporcional al tamaño del step:
        # para un step de 10 cm usa T_RAMP_PER_10CM segundos.
        if ramp_active:
            step_size   = abs(yr_target - yr_ramp_start)          # m
            if step_size > 1e-4:
                ramp_rate = step_size / (T_RAMP_PER_10CM * (step_size / 0.10))
                # → siempre T_RAMP_PER_10CM s independiente del tamaño del step
                ramp_rate = 0.10 / T_RAMP_PER_10CM                # m/s fijo
                delta_ramp = ramp_rate * Ts
                if yr_target > yr_ramp_start:
                    yr_ramp = min(yr_ramp + delta_ramp, yr_target)
                else:
                    yr_ramp = max(yr_ramp - delta_ramp, yr_target)
            if abs(yr_ramp - yr_target) < 1e-4:
                yr_ramp   = yr_target
                ramp_active = False
                print(f"\n  ✓ Rampa completada en t={k*Ts:.0f}s → ref={yr_target*100:.1f}cm")
        else:
            yr_ramp = yr_target

        # ── Contador de transitorio ─────────────────────────────────────────
        transient_counter += 1
        if in_transient and transient_counter > TRANSIENT_STEPS:
            in_transient = False
            print(f"\n  ✓ Transitorio terminado en t={k*Ts:.0f}s "
                  f"(paso {k}) — integrador activo")

        # 1. Predicción Koopman
        z_koopman_pred = A_d @ z_aug[:nz] + B_d @ uprev
        z_koopman_pred[:3] = np.maximum(z_koopman_pred[:3], 0)
        z_koopman_pred[6:] = np.maximum(z_koopman_pred[6:], 0)

        # 2. Leer medición
        h1_meas, h2_real, h3_meas = esp.get_measurement(timeout=Ts * 2)
        H_meas[0, k+1]  = h1_meas
        H_meas[1, k+1]  = h3_meas
        H2_REAL[k+1]    = h2_real

        # 3. Corrección Luenberger — solo h1 y h3
        y_obs = np.array([
            np.sqrt(max(h1_meas, EPS_PSI)),
            np.sqrt(max(h3_meas, EPS_PSI)),
            h1_meas,
            h3_meas,
        ])
        innov     = y_obs - C_obs @ z_koopman_pred
        z_koopman = z_koopman_pred + L_d @ innov
        z_koopman[:3] = np.maximum(z_koopman[:3], 0)
        z_koopman[6:] = np.maximum(z_koopman[6:], 0)

        # 4. Estado físico estimado
        x_hat  = np.clip(z_koopman[:3]**2, 0, 0.75)
        h2_est = x_hat[1]
        X_hat[:, k+1] = x_hat

        # ── CORRECCIÓN CLAVE: integrador con freeze en transitorio ──────────
        # Durante el transitorio (in_transient=True) el integrador se congela
        # completamente, evitando el windup que genera el sobrepico.
        # En estado estacionario aplica el deadband original.
        # El integrador usa yr_ramp (no yr) para no acumularse antes de llegar.
        error_h2 = yr_ramp - h2_est
        if not in_transient and not ramp_active:
            if abs(error_h2) < DEADBAND_H2:
                z_int += error_h2 * Ts
                z_int  = np.clip(z_int, -Z_INT_LIMIT, Z_INT_LIMIT)
            # Si |error| >= DEADBAND pero no en transitorio: freeze (picos de ruido)
        # Si in_transient o ramp_active: z_int no cambia (freeze total)
        # ────────────────────────────────────────────────────────────────────
        Z_INT[k+1] = z_int

        # 5. Estado aumentado
        z_aug = np.append(z_koopman, z_int)

        # 6. KoopmanMPC — usa yr_ramp como referencia interna durante la rampa
        if w0_mpc is None:
            w0_mpc = np.zeros(nz_aug*(N_mpc+1) + nu*N_mpc)
            w0_mpc[:nz_aug] = z_aug

        par_val = np.concatenate([z_aug, [yr_ramp], uprev])
        try:
            sol    = solver(x0=w0_mpc, lbx=lbw, ubx=ubw,
                            lbg=lbg, ubg=ubg, p=par_val)
            w_opt  = np.array(sol['x']).flatten()
            Z_opt  = w_opt[:nz_aug*(N_mpc+1)].reshape(N_mpc+1, nz_aug)
            U_opt  = w_opt[nz_aug*(N_mpc+1):].reshape(N_mpc, nu)
            u_k    = np.clip(U_opt[0], umin, umax)
            Z_sh   = np.vstack([Z_opt[1:], Z_opt[-1:]])
            U_sh   = np.vstack([U_opt[1:], U_opt[-1:]])
            w0_mpc = np.concatenate([Z_sh.flatten(), U_sh.flatten()])
        except Exception as e:
            print(f"  ⚠ MPC falló en k={k}: {e} — usando u anterior")
            u_k = uprev.copy(); w0_mpc = None

        elapsed_ms = (time.time() - t_step_start) * 1000
        step_ms.append(elapsed_ms)
        U_hist[:, k] = u_k; uprev = u_k.copy()

        esp.send_control(u_k[0], u_k[1])

        if k % 5 == 0:
            e_obs  = (h2_est - h2_real) * 100
            trns   = 'SI' if in_transient else 'NO'
            ramp_s = f'R={yr_ramp*100:.1f}' if ramp_active else f'{yr*100:.1f}'
            print(f"  {k*Ts:>6.0f} | {h1_meas*100:>7.2f} | {h2_est*100:>7.2f} | "
                  f"{h2_real*100:>7.2f} | {e_obs:>+8.3f} | {ramp_s:>7} | "
                  f"{u_k[0]:>5.3f} | {elapsed_ms:>5.0f} | {trns:>5}")

        dt_used = time.time() - t_step_start
        if dt_used < Ts * 0.9:
            time.sleep(Ts - dt_used)

    print(f"\n  Experimento finalizado — {(time.time()-t_experiment_start)/60:.1f} min")
    print(f"  Outliers rechazados: {esp.outlier_count}")

    return {
        'T_hist'            : T_hist,
        'H_meas'            : H_meas,
        'H2_REAL'           : H2_REAL,
        'X_hat'             : X_hat,
        'Z_INT'             : Z_INT,
        'U_hist'            : U_hist,
        'yref_v'            : yref_v,
        'step_ms'           : step_ms,
        'ref_change_times_s': ref_change_times_s,
        'outliers'          : esp.outlier_count,
    }

# ============================================================================
# MÉTRICAS
# ============================================================================
def compute_metrics(res):
    T   = res['T_hist']
    ref = np.append(res['yref_v'], res['yref_v'][-1])
    rct = res['ref_change_times_s']

    idx_400 = T > 400
    settling_mask = np.ones(len(T), dtype=bool)
    for tc in rct:
        settling_mask &= ~((T >= tc) & (T < tc + SETTLING_S))
    mask_ss = idx_400 & settling_mask

    e_h2_ctrl = (res['X_hat'][1, :] - ref) * 100
    e_h1_est  = (res['X_hat'][0, :] - res['H_meas'][0, :]) * 100
    e_h3_est  = (res['X_hat'][2, :] - res['H_meas'][1, :]) * 100

    rmse_ctrl  = float(np.sqrt(np.mean(e_h2_ctrl[mask_ss]**2)))
    rmse_h1    = float(np.sqrt(np.mean(e_h1_est[mask_ss]**2)))
    rmse_h3    = float(np.sqrt(np.mean(e_h3_est[mask_ss]**2)))
    bias_ctrl  = float(np.mean(e_h2_ctrl[mask_ss]))
    mae_ctrl   = float(np.mean(np.abs(e_h2_ctrl[mask_ss])))
    rmse_trans = float(np.sqrt(np.mean(e_h2_ctrl[idx_400]**2)))

    e_obs = (res['X_hat'][1, :] - res['H2_REAL']) * 100
    rmse_obs_ss  = float(np.sqrt(np.mean(e_obs[mask_ss]**2)))
    bias_obs_ss  = float(np.mean(e_obs[mask_ss]))
    mae_obs_ss   = float(np.mean(np.abs(e_obs[mask_ss])))

    return {
        'rmse_ctrl'  : rmse_ctrl,
        'rmse_h1'    : rmse_h1,
        'rmse_h3'    : rmse_h3,
        'bias_ctrl'  : bias_ctrl,
        'mae_ctrl'   : mae_ctrl,
        'rmse_trans' : rmse_trans,
        'e_h2_ctrl'  : e_h2_ctrl,
        'mask_ss'    : mask_ss,
        'e_obs'      : e_obs,
        'rmse_obs_ss': rmse_obs_ss,
        'bias_obs_ss': bias_obs_ss,
        'mae_obs_ss' : mae_obs_ss,
    }

# ============================================================================
# GRÁFICAS
# ============================================================================
def plot_results(res, save_dir=output_dir):
    T   = res['T_hist']
    ref = np.append(res['yref_v'], res['yref_v'][-1])
    rct = res['ref_change_times_s']
    m   = compute_metrics(res)

    avg_ms = np.mean(res['step_ms'])
    p99_ms = np.percentile(res['step_ms'], 99)

    fig, axes = plt.subplots(4, 2, figsize=(14, 16))
    fig.suptitle(
        f'Koopman Observer+MPC v4b-fix — Planta Real | '
        f'{datetime.now().strftime("%Y-%m-%d %H:%M")}\n'
        f'RMSE_ctrl={m["rmse_ctrl"]:.3f}cm | Bias_ctrl={m["bias_ctrl"]:.3f}cm | '
        f'RMSE_obs={m["rmse_obs_ss"]:.3f}cm | Bias_obs={m["bias_obs_ss"]:.3f}cm | '
        f'{avg_ms:.0f}ms/paso',
        fontsize=10, fontweight='bold')

    ax = axes[0, 0]
    ax.plot(T, res['H_meas'][0]*100, 'b',   lw=1.5, label='$h_1$ medido')
    ax.plot(T, res['X_hat'][0]*100,  'r--', lw=1.5,
            label=f'$h_1$ Koopman (RMSE={m["rmse_h1"]:.3f}cm)')
    ax.set_ylabel('$h_1$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 1 (medido)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    ax.plot(T, res['H_meas'][1]*100, 'b',   lw=1.5, label='$h_3$ medido')
    ax.plot(T, res['X_hat'][2]*100,  'r--', lw=1.5,
            label=f'$h_3$ Koopman (RMSE={m["rmse_h3"]:.3f}cm)')
    ax.set_ylabel('$h_3$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 3 (medido)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    ax.plot(T, res['H2_REAL']*100,  'g',   lw=1.8,
            label='$h_2$ real (sensor verificación)')
    ax.plot(T, res['X_hat'][1]*100, 'r--', lw=2.0,
            label=f'$h_2$ estimado Koopman (RMSE_obs={m["rmse_obs_ss"]:.3f}cm)')
    ax.stairs(ref[:-1]*100, T, color='k', linestyle='--', lw=1.5, label='Referencia')
    for tc in rct:
        ax.axvline(tc, color='gray', lw=0.8, ls=':')
    ax.set_ylabel('$h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Tanque 2 — estimado Koopman vs. sensor verdad-terreno')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    ax.plot(T, m['e_obs'], color='darkgreen', lw=1.5,
            label=(f'$e_{{obs}}$ = $\\hat{{h}}_2$ − $h_2^{{real}}$ | '
                   f'RMSE={m["rmse_obs_ss"]:.3f}cm | '
                   f'Bias={m["bias_obs_ss"]:.3f}cm'))
    ax.axhline(0, color='k', ls=':')
    ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
    for i, tc in enumerate(rct):
        ax.axvspan(tc, tc+SETTLING_S, alpha=0.12, color='orange',
                   label='transitorio excluido' if i == 0 else None)
    ax.set_ylabel('Error observador [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Error puro del observador Koopman (estimado − real)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[2, 0]
    ax.plot(T, m['e_h2_ctrl'], 'r', lw=1.5,
            label=(f'RMSE_SS={m["rmse_ctrl"]:.3f}cm | '
                   f'RMSE_trans={m["rmse_trans"]:.3f}cm | '
                   f'Bias={m["bias_ctrl"]:.3f}cm'))
    ax.axhline(0, color='k', ls=':')
    ax.axvline(400, color='gray', lw=0.8, ls=':', label='t=400s')
    for i, tc in enumerate(rct):
        ax.axvspan(tc, tc+SETTLING_S, alpha=0.12, color='orange',
                   label='transitorio excluido' if i == 0 else None)
    ax.set_ylabel('Error ctrl $h_2$ [cm]'); ax.set_xlabel('t [s]')
    ax.set_title('Error de seguimiento (control)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[2, 1]
    ax.stairs(res['U_hist'][0], T, color='b', lw=1.5, label='$u_1$')
    ax.stairs(res['U_hist'][1], T, color='r', lw=1.5, label='$u_3$')
    ax2 = ax.twinx()
    ax2.plot(T, res['Z_INT']*100, 'g--', lw=1, alpha=0.7, label='$z_{int}$ [cm·s]')
    ax2.axhline( DEADBAND_H2*100, color='gray', lw=0.8, ls=':', alpha=0.6)
    ax2.axhline(-DEADBAND_H2*100, color='gray', lw=0.8, ls=':', alpha=0.6)
    ax2.set_ylabel('$z_{int}$ [cm·s]', color='g')
    ax2.set_ylim([-Z_INT_LIMIT*100*1.2, Z_INT_LIMIT*100*1.2])
    ax.set_ylim([-0.05, 1.1]); ax.set_ylabel('Control [0–1]')
    ax.set_xlabel('t [s]'); ax.set_title('Entradas KoopmanMPC v4b-fix + integrador')
    lines1, lbl1 = ax.get_legend_handles_labels()
    lines2, lbl2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, lbl1 + lbl2, fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[3, 0]
    ax.plot(res['step_ms'], 'k', lw=0.8, alpha=0.7)
    ax.axhline(avg_ms, color='b', lw=1.5, ls='--', label=f'Media={avg_ms:.0f}ms')
    ax.axhline(p99_ms, color='r', lw=1.5, ls=':', label=f'P99={p99_ms:.0f}ms')
    ax.axhline(Ts*1000, color='g', lw=1.5, ls='-', label=f'Ts={Ts*1000:.0f}ms')
    ax.set_ylabel('Tiempo [ms]'); ax.set_xlabel('Paso k')
    ax.set_title('Tiempo de cómputo por paso'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    ax = axes[3, 1]
    ax.axis('off')
    tabla = [
        ['Métrica', 'Koopman v4b-fix', 'v4b original', 'MHE+NMPC'],
        ['RMSE ctrl h₂ SS [cm]',    f'{m["rmse_ctrl"]:.3f}',   '0.281', '0.298'],
        ['MAE  ctrl h₂ SS [cm]',    f'{m["mae_ctrl"]:.3f}',    '0.234', '0.262'],
        ['Bias ctrl h₂ SS [cm]',    f'{m["bias_ctrl"]:.3f}',   '0.006', '−0.258'],
        ['─────────────', '──────────', '──────────', '──────────'],
        ['RMSE obs h₂ SS [cm]',     f'{m["rmse_obs_ss"]:.3f}', '0.557', '—'],
        ['Bias obs h₂ SS [cm]',     f'{m["bias_obs_ss"]:.3f}', '−0.291', '—'],
        ['─────────────', '──────────', '──────────', '──────────'],
        ['RMSE est. h₁ SS [cm]',    f'{m["rmse_h1"]:.3f}',     '0.644', '—'],
        ['RMSE est. h₃ SS [cm]',    f'{m["rmse_h3"]:.3f}',     '0.657', '—'],
        ['Tiempo medio [ms]',        f'{avg_ms:.1f}',           '2099', '~20'],
    ]
    t = ax.table(cellText=tabla[1:], colLabels=tabla[0],
                 loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(8)
    t.scale(1.1, 1.4)
    ax.set_title('Comparación de esquemas', fontsize=10, fontweight='bold', pad=10)

    plt.tight_layout()
    tstamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    fname  = save_dir / f'koopman_v4b_fix_{tstamp}.png'
    plt.savefig(fname, dpi=200, bbox_inches='tight'); plt.close()
    print(f"\n  Figura guardada: {fname}")

    csv_path = save_dir / f'koopman_v4b_fix_{tstamp}.csv'
    with open(csv_path, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['t_s', 'h1_meas_m', 'h1_est_m',
                    'h2_est_m', 'h2_real_m', 'error_obs_cm',
                    'h3_meas_m', 'h3_est_m',
                    'ref_m', 'u1', 'u3', 'z_int', 'error_ctrl_cm'])
        for k in range(Nsim):
            e_ctrl = (res['X_hat'][1, k] - res['yref_v'][k]) * 100
            e_obs  = (res['X_hat'][1, k] - res['H2_REAL'][k]) * 100
            w.writerow([
                T[k],
                res['H_meas'][0, k],  res['X_hat'][0, k],
                res['X_hat'][1, k],   res['H2_REAL'][k],  e_obs,
                res['H_meas'][1, k],  res['X_hat'][2, k],
                res['yref_v'][k],
                res['U_hist'][0, k],  res['U_hist'][1, k],
                res['Z_INT'][k],      e_ctrl,
            ])
    print(f"  Datos guardados: {csv_path}")

# ============================================================================
# RESUMEN EN CONSOLA
# ============================================================================
def print_summary(res):
    m      = compute_metrics(res)
    avg_ms = np.mean(res['step_ms'])
    p99_ms = np.percentile(res['step_ms'], 99)

    print(f"\n  {'='*60}")
    print(f"  RESUMEN KOOPMAN OBSERVER+MPC v4b-fix — PLANTA REAL")
    print(f"  KI={KI_WEIGHT} | Z_INT_LIMIT=±{Z_INT_LIMIT*100:.1f}cm·s | "
          f"DEADBAND={DEADBAND_H2*100:.0f}cm")
    print(f"  Anti-windup: freeze durante transitorio ({30*Ts:.0f}s)")
    print(f"  {'='*60}")
    print(f"\n  MÉTRICAS DE CONTROL (ĥ₂ vs. referencia):")
    print(f"  {'RMSE seguimiento h₂ (SS):':<40} {m['rmse_ctrl']:.4f} cm")
    print(f"  {'RMSE seguimiento h₂ (transitorio):':<40} {m['rmse_trans']:.4f} cm")
    print(f"  {'MAE  seguimiento h₂ (SS):':<40} {m['mae_ctrl']:.4f} cm")
    print(f"  {'Bias seguimiento h₂ (SS):':<40} {m['bias_ctrl']:.4f} cm")
    print(f"\n  MÉTRICAS DEL OBSERVADOR (ĥ₂ vs. h₂ real):")
    print(f"  {'RMSE estimación h₂ (SS):':<40} {m['rmse_obs_ss']:.4f} cm")
    print(f"  {'Bias estimación h₂ (SS):':<40} {m['bias_obs_ss']:.4f} cm")
    print(f"  {'MAE  estimación h₂ (SS):':<40} {m['mae_obs_ss']:.4f} cm")
    print(f"\n  ESTIMACIÓN h₁ y h₃:")
    print(f"  {'RMSE estimación h₁ (SS):':<40} {m['rmse_h1']:.4f} cm")
    print(f"  {'RMSE estimación h₃ (SS):':<40} {m['rmse_h3']:.4f} cm")
    print(f"\n  TIEMPO DE CÓMPUTO:")
    print(f"  {'Tiempo medio por paso:':<40} {avg_ms:.1f} ms")
    print(f"  {'Tiempo p99  por paso:':<40} {p99_ms:.1f} ms")
    print(f"  {'Outliers rechazados:':<40} {res['outliers']}")
    print(f"  {'='*60}")

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 68)
    print("  Koopman Closed-Loop v4b-fix — PLANTA REAL")
    print(f"  Puerto: {SERIAL_PORT} | Ts={Ts}s | T_sim={T_sim}s")
    print(f"  Anti-windup: freeze integrador durante transitorio")
    print(f"  KI_WEIGHT={KI_WEIGHT} | Z_INT_LIMIT=±{Z_INT_LIMIT*100:.1f}cm·s")
    print("=" * 68)

    print("\nCalculando puntos de operación nominales...")
    SS_NOM = {}
    for h2r in [0.10, 0.15, 0.20]:
        x_ss, u_ss = compute_ss(h2r)
        if x_ss is not None:
            SS_NOM[h2r] = (x_ss, u_ss)
            print(f"  h2={h2r:.2f}m → x_ss={np.round(x_ss,4)} | u_ss={np.round(u_ss,4)} ✓")
        else:
            print(f"  h2={h2r:.2f}m → ✗ No convergió")

    if not SS_NOM:
        print("\n  ERROR: No se encontró ningún punto estacionario.")
        sys.exit(1)

    A_d, B_d, nz, nu = train_edmd(verbose=True)
    L_d, C_obs = design_observer(A_d, nz)
    mpc_args   = build_mpc_with_integrator(A_d, B_d, nz, nu)
    solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg, nz_aug, nu2 = mpc_args

    esp = ESP32Interface(port=SERIAL_PORT, baud=BAUD_RATE)
    try:
        esp.connect()

        print("\n  Verificación de sensores (5 lecturas):")
        for i in range(5):
            h1, h2r, h3 = esp.get_measurement()
            print(f"    [{i+1}] h1={h1*100:.2f}cm | h2_real={h2r*100:.2f}cm | h3={h3*100:.2f}cm")
            time.sleep(0.5)

        input("\n  ¿Todo OK? Presiona ENTER para iniciar (Ctrl+C para cancelar)...")

        res = run_real_experiment(
            esp, A_d, B_d, L_d, C_obs,
            solver, N_mpc, umin, umax, lbw, ubw, lbg, ubg,
            nz_aug, nu2, nz, SS_NOM
        )

        print_summary(res)
        plot_results(res)

    except KeyboardInterrupt:
        print("\n\n  Interrumpido por el usuario.")
    except Exception as e:
        print(f"\n  ERROR: {e}")
        import traceback; traceback.print_exc()
    finally:
        esp.close()

if __name__ == '__main__':
    main()

  Koopman Closed-Loop v4b-fix — PLANTA REAL
  Puerto: COM7 | Ts=2.0s | T_sim=800s
  Anti-windup: freeze integrador durante transitorio
  KI_WEIGHT=80.0 | Z_INT_LIMIT=±1.5cm·s

Calculando puntos de operación nominales...
  h2=0.10m → x_ss=[0.1139 0.1    0.0818] | u_ss=[0.3283 0.0012] ✓
  h2=0.15m → x_ss=[0.1675 0.15   0.1269] | u_ss=[0.4029 0.0012] ✓
  h2=0.20m → x_ss=[0.2207 0.2    0.1727] | u_ss=[0.4659 0.0012] ✓

Entrenando EDMD (parámetros nominales)...
  8220 muestras de entrenamiento
  RMSE one-step: 9.74e-05
  |λ| ∈ [0.03444, 0.99990] ✓

Diseñando ganancia L (DARE con Q=0.1*I)...
  q=0.1: max|λ(A-LC)|=0.95481, max|L|=0.2440 ✓
  τ_decay = 43.3 s

Construyendo KoopmanMPC v4b-fix con integrador (ki=80.0)...
  KoopmanMPC v4b-fix ✓  (nz_aug=9)
  DEADBAND_H2=3.0cm | Z_INT_LIMIT=±1.5cm·s
  Peso terminal: wk=5 en últimos 3 pasos

Conectando a COM7 @ 115200 baud...
  ✓ Conectado a COM7

  Verificación de sensores (5 lecturas):
    [1] h1=3.03cm | h2_real=1.86cm | h3=2.49cm
    [2] h1=3.02